# 00 · Preamble / Environment Setup

**Purpose:** one-time “power-on” cell that standardizes paths, config, randomness, plotting, and helper imports for the whole project. It keeps all other notebooks short and consistent.

### What this notebook does

* **Finds the project root** (walks up until it sees `configs/params.yaml`) and loads it.
* **Echoes key paths** (where the `.h5ad` is, where figures/tables will be saved) and **creates those folders** if missing.
* **Sets seeds & caps CPU threads** (reproducible results; no noisy oversubscription).
* **Sets Matplotlib defaults** and defines a tiny `savefig("figXX.png")` helper.
* **Adds** `notebooks/_lib` **to `sys.path`** and **smoke-tests** helper functions.
* **Loads & checks marker lists** (`S_mouse.txt`, `G2M_mouse.txt`) and prints counts.

### How to use it

1. Select the **`LimitCycle (mouse GSE154989)`** kernel for this notebook.
2. **Run all cells once per session** (or after changing `configs/params.yaml`).
3. Proceed to `01_ingest_build_anndata.ipynb` → `02_...` in order.

### Expected console prints

* `Repo root: .../gp1_limitcycle_mm_gse154989`
* `AnnData path: .../data/interim/mm_timecourse.h5ad`
* `[seed] set to 42` and thread caps dict
* `[helpers] loaded from .../notebooks/_lib`
* `[markers] S=…  G2M=…`

### Quick checks if something breaks

* Ensure `configs/params.yaml` exists and has:

  * `paths.anndata`, `paths.figures`, `paths.tables`
  * `markers.s_file`, `markers.g2m_file`
* Confirm marker files exist at those paths.
* If the root isn’t found, make sure you opened the notebook **inside** the project folder.


### Aim
Locate the project root, load `configs/params.yaml`, and materialize output directories so all downstream notebooks share the same paths and parameters.

Inputs: filesystem (expects `configs/params.yaml`)
Method: walk up to find the repo root; parse YAML with safe loader; fill sane defaults for missing keys; create `outputs/figures` and `outputs/tables`.
Parameters recorded: `paths.*`, `markers.*`, `model.seed`
Acceptance check: `configs/params.yaml` found and both output directories exist after this cell.


In [1]:
# --- Cell 1: paths + params (robust repo root + friendly checks) ---
from pathlib import Path
import yaml

def repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    # climb up to find configs/params.yaml
    for _ in range(8):
        if (p / "configs" / "params.yaml").exists():
            return p
        p = p.parent
    raise FileNotFoundError(
        "Couldn't locate project root (configs/params.yaml). "
        "Open this notebook inside the project folder."
    )

BASE = repo_root()
PARAMS = BASE / "configs" / "params.yaml"

# load YAML safely
with open(PARAMS, "r") as fh:
    P = yaml.safe_load(fh) or {}

# provide sane defaults if keys are missing
P.setdefault("paths", {})
P["paths"].setdefault("anndata", "data/interim/mm_timecourse.h5ad")
P["paths"].setdefault("figures", "outputs/figures")
P["paths"].setdefault("tables",  "outputs/tables")

P.setdefault("markers", {})
P["markers"].setdefault("s_file",  "configs/markers/S_mouse.txt")
P["markers"].setdefault("g2m_file","configs/markers/G2M_mouse.txt")

P.setdefault("model", {})
P["model"].setdefault("seed", 42)

# unpack for convenience
PATHS = P["paths"]; MODEL = P["model"]; MARK = P["markers"]

# ensure output dirs exist
(fig_dir := BASE / PATHS["figures"]).mkdir(parents=True, exist_ok=True)
(tab_dir := BASE / PATHS["tables"]).mkdir(parents=True, exist_ok=True)

# echo what we’re using
print("Repo root      :", BASE)
print("Params file    :", PARAMS)
print("AnnData path   :", BASE / PATHS["anndata"])
print("Figures dir    :", fig_dir)
print("Tables dir     :", tab_dir)
print("S markers file :", BASE / MARK["s_file"])
print("G2M markers    :", BASE / MARK["g2m_file"])
print("Seed           :", MODEL["seed"])


Repo root      : /home/secondbook5/JHU_Bioinformatics/SystemsBiology/luad-limitcycle-landscape-flux
Params file    : /home/secondbook5/JHU_Bioinformatics/SystemsBiology/luad-limitcycle-landscape-flux/configs/params.yaml
AnnData path   : /home/secondbook5/JHU_Bioinformatics/SystemsBiology/luad-limitcycle-landscape-flux/data/interim/mm_timecourse.h5ad
Figures dir    : /home/secondbook5/JHU_Bioinformatics/SystemsBiology/luad-limitcycle-landscape-flux/outputs/figures
Tables dir     : /home/secondbook5/JHU_Bioinformatics/SystemsBiology/luad-limitcycle-landscape-flux/outputs/tables
S markers file : /home/secondbook5/JHU_Bioinformatics/SystemsBiology/luad-limitcycle-landscape-flux/configs/markers/S_mouse.txt
G2M markers    : /home/secondbook5/JHU_Bioinformatics/SystemsBiology/luad-limitcycle-landscape-flux/configs/markers/G2M_mouse.txt
Seed           : 42


### Result
Project root, parameter file, and key paths were resolved and echoed. Output directories now exist.

Artifacts: none created yet (directories only)
Notes: subsequent notebooks should read paths from `P`/`PATHS` rather than hardcoding.
<!-- PAPER HOOK: Analyses were parameterized via a single YAML and executed from a fixed repository root to ensure reproducibility. -->


### Aim
Set deterministic execution conditions by capping BLAS/OpenMP threads and seeding Python and NumPy RNGs.

Inputs: `MODEL["seed"]` from YAML
Method: set thread caps for OMP/OPENBLAS/MKL/NUMEXPR before importing NumPy; seed Python's `random` and NumPy.
Parameters recorded: thread cap = 4, seed = `MODEL["seed"]`
Acceptance check: console prints show all four thread env vars set and `[seed] set to <value>`.


In [2]:
# --- Cell 2: threads first, then seeds (so BLAS sees the caps) ---
import os

def set_cpu_threads(n: int = 4, override: bool = False):
    envs = ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]
    for k in envs:
        if override or os.environ.get(k) is None:
            os.environ[k] = str(n)
    print("[threads]", {k: os.environ.get(k) for k in envs})

set_cpu_threads(4, override=False)

# now it's safe to import numpy & friends
import random, numpy as np

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    print(f"[seed] set to {seed}")

set_seed(int(MODEL.get("seed", 42)))


[threads] {'OMP_NUM_THREADS': '4', 'OPENBLAS_NUM_THREADS': '4', 'MKL_NUM_THREADS': '4', 'NUMEXPR_NUM_THREADS': '4'}
[seed] set to 42


### Result
Thread caps applied to OMP/OPENBLAS/MKL/NUMEXPR and RNG seed set to 42.

Artifacts: none
Notes: this cell should run once per kernel session to keep results stable across runs.
<!-- PAPER HOOK: Numerical experiments were executed with fixed seeds and controlled thread counts to avoid nondeterminism from BLAS parallelism. -->


### Aim
Standardize figure appearance and define a single helper for saving figures.

Inputs: `BASE / PATHS["figures"]`
Method: set Matplotlib rcParams for DPI and axes; define `savefig(name, tight=True)` that writes to the figures directory and logs the path.
Parameters recorded: `figure.dpi=120`, `savefig.dpi=200`
Acceptance check: `savefig` is defined and points to `outputs/figures`.


In [3]:
# --- Cell 3: plotting defaults + save helper ---
import matplotlib.pyplot as plt

# modest, readable defaults
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.grid"] = False

FIGDIR = BASE / PATHS["figures"]

def savefig(name: str, *, tight: bool = True):
    """
    Save current Matplotlib figure into outputs/figures/.
    Usage: plt.plot(...); savefig("fig01_s_vs_g2m_circle.png")
    """
    out = FIGDIR / name
    if tight:
        plt.tight_layout()
    plt.savefig(out, bbox_inches="tight")
    print("[saved]", out)


### Result
Matplotlib defaults configured and `savefig()` is available for consistent figure export.

Artifacts: none yet (helper only)
Notes: use `savefig("figXX_name.png")` immediately after plotting to enforce uniform DPI and paths.
<!-- PAPER HOOK: All figures were emitted through a single helper with fixed DPI and paths to ensure consistent rendering. -->


### Aim
Expose the project helper library and verify availability of core utility functions used by later notebooks.

Inputs: `notebooks/_lib/cc_limitcycle.py`
Method: prepend `_lib` to `sys.path`; import `cc_limitcycle`; assert presence of expected functions by name.
Acceptance check: import succeeds and all listed utilities report `True`.


In [4]:
# --- Cell 4: add notebooks/_lib to sys.path and smoke-test helpers ---
import sys
LIB = BASE / "notebooks" / "_lib"
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))

try:
    import cc_limitcycle as cc
    print("[helpers] loaded from", LIB)
    # show key functions we expect to use
    expected = [
        "fit_circle",
        "angles_from_center",
        "orient_phases",
        "annulus_disk_gmm",
        "vm_kde",
        "select_kappa_cv",
        "ring_density_and_potential",
        "find_barriers",
        "relative_speed_from_density",
        "fit_periodic",
        "circular_w1_hist",
        "piecewise_warp",
    ]
    for name in expected:
        print(f"  - {name:30s}", getattr(cc, name, None) is not None)
except Exception as e:
    print("[helpers] not loaded:", repr(e))


[helpers] loaded from /home/secondbook5/JHU_Bioinformatics/SystemsBiology/luad-limitcycle-landscape-flux/notebooks/_lib
  - fit_circle                     True
  - angles_from_center             True
  - orient_phases                  True
  - annulus_disk_gmm               True
  - vm_kde                         True
  - select_kappa_cv                True
  - ring_density_and_potential     True
  - find_barriers                  True
  - relative_speed_from_density    True
  - fit_periodic                   True
  - circular_w1_hist               True
  - piecewise_warp                 True


### Result
Helper module loaded from `notebooks/_lib` and expected utilities are present.

Artifacts: none
Notes: central utilities (circle fit, circular KDE, barrier finding, W1, etc.) will be called by analysis notebooks.
<!-- PAPER HOOK: A shared helper library implements circle fitting, circular density, barrier calls, and circular Wasserstein to keep analyses consistent. -->


### Aim
Verify presence of S and G2M marker lists and load them into memory for later checks.

Inputs: `configs/markers/S_mouse.txt`, `configs/markers/G2M_mouse.txt`
Method: assert files exist; simple line-based loader that ignores blanks and comments.
Acceptance check: both files exist and counts > 0 for each list.


In [5]:
# --- Cell 5: verify marker files & define a tiny loader ---
from pathlib import Path

S_FILE   = BASE / MARK["s_file"]
G2M_FILE = BASE / MARK["g2m_file"]

def load_markers(path: Path) -> list[str]:
    return [ln.strip() for ln in open(path, "r") if ln.strip() and not ln.startswith("#")]

assert S_FILE.exists(),  f"Missing S markers file: {S_FILE}"
assert G2M_FILE.exists(),f"Missing G2M markers file: {G2M_FILE}"

S_genes   = load_markers(S_FILE)
G2M_genes = load_markers(G2M_FILE)
print(f"[markers] S={len(S_genes)}  G2M={len(G2M_genes)}")
print(" sample S:", S_genes[:5])
print(" sample G2M:", G2M_genes[:5])


[markers] S=41  G2M=46
 sample S: ['Mcm5', 'Pcna', 'Tyms', 'Fen1', 'Mcm2']
 sample G2M: ['Hmgb2', 'Cdk1', 'Nusap1', 'Ube2c', 'Birc5']


### Result
Loaded 41 S-phase and 46 G2M markers; printed the first few entries for each.

Artifacts: none
Notes: these lists will be used to sanity-check ring orientation and cycling fractions.
<!-- PAPER HOOK: Mouse S and G2M marker sets (41 and 46 genes, respectively) were used for orientation checks and downstream validation. -->


### Aim
Record core software versions and confirm that the expected AnnData file is present.

Inputs: library imports; `BASE / PATHS["anndata"]`
Method: import and print versions of numpy, pandas, scanpy, anndata, sklearn, matplotlib; check existence of the `.h5ad`.
Acceptance check: versions printed and AnnData path exists (or a clear message indicates where it will be created).


In [6]:
# --- Cell 6: environment versions & critical file presence ---
import importlib, platform
mods = ["numpy", "pandas", "scanpy", "anndata", "sklearn", "matplotlib"]
vers = {m: importlib.import_module(m).__version__ for m in mods if importlib.util.find_spec(m)}
print("[python]", platform.python_version())
print("[versions]", vers)

ANN = BASE / PATHS["anndata"]
print("[check] AnnData exists? ", ANN.exists(), "->", ANN)
if not ANN.exists():
    print("         (You’ll create this in 01_ingest_build_anndata.ipynb)")


[python] 3.12.2
[versions] {'numpy': '2.3.3', 'pandas': '2.3.3', 'scanpy': '1.11.4', 'anndata': '0.12.2', 'sklearn': '1.7.2', 'matplotlib': '3.10.6'}
[check] AnnData exists?  True -> /home/secondbook5/JHU_Bioinformatics/SystemsBiology/luad-limitcycle-landscape-flux/data/interim/mm_timecourse.h5ad


### Result
Environment: Python 3.12.2; numpy 2.3.3; pandas 2.3.3; scanpy 1.11.4; anndata 0.12.2; sklearn 1.7.2; matplotlib 3.10.6. AnnData file present at the configured path.

Artifacts: none
Notes: if the `.h5ad` is absent, generate it in `01_ingest_build_anndata.ipynb`.
<!-- PAPER HOOK: Analyses were run with pinned versions (Python 3.12.2; Scanpy 1.11.4; AnnData 0.12.2), and the curated `.h5ad` was available at the configured path. -->
